***Character-Level Tokenizer (stoi/itos/BOS)***

In [ ]:
class CharTokenizer:
	def __init__(self, text: str):
		
		special = ['<BOS>', '<EOS>']
		chars = sorted(set(text))
		vocab = special + chars
		self.stoi = {ch: i for i, ch in enumerate(vocab)}   # char → index
		self.itos = {i: ch for i, ch in enumerate(vocab)}   # index → char
		self.vocab_size = len(vocab)

	def encode(self, text: str) -> list:
		
		return [self.stoi['<BOS>']] + [self.stoi[ch] for ch in text] + [self.stoi['<EOS>']]
		
	def decode(self, indices: list) -> str:
		
		return "".join([self.itos[i] for i in indices])

***Random Train/Validation/Test Split with Shuffling***

In [ ]:
import numpy as np

def random_split(data: np.ndarray, train_frac: float, validation_frac: float, seed: int = 123) -> list:
	"""
	Randomly split a dataset into train, validation, and test subsets.
	"""
	
	data = np.array(data)
	n = len(data)          # n = 10

	# Shuffle indices
	indices = np.random.default_rng(seed).permutation(n)    # ex : indices = [7 4 0 2 1 5 6 8 3 9]
	
	train_end = int(n * train_frac)                         # train_end = (10 * 0.7) = 7
	validation_end = train_end + int(n * validation_frac)   # validation_end = 7 + (10 * 0.1) = 8
	
	train = data[indices[:train_end]]                       # train = data[indices[:7]] = data[7, 4, 0, 2, 1, 5, 6] 
	validation = data[indices[train_end:validation_end]]    # validation = data[indices[7 : 8]] = data[6] 
	test = data[indices[validation_end:]]                   # test = data[indices[8:]] = data[8, 3, 9]

	return [train, validation, test]

data = np.arange(20).reshape(10, 2)
train, val, test = random_split(data, 0.7, 0.1, seed=123)
print([train.tolist(), val.tolist(), test.tolist()])

[[[14, 15], [8, 9], [0, 1], [4, 5], [2, 3], [10, 11], [12, 13]], [[16, 17]], [[6, 7], [18, 19]]]


In [7]:
data = np.arange(20).reshape(10, 2)
data

array([[ 0,  1],
       [ 2,  3],
       [ 4,  5],
       [ 6,  7],
       [ 8,  9],
       [10, 11],
       [12, 13],
       [14, 15],
       [16, 17],
       [18, 19]])

***Generate Input-Target Batches for Language Model Training***

In [ ]:
import numpy as np

def get_batch(data: np.ndarray, block_size: int, batch_size: int, seed: int) -> np.ndarray:
	# Your code here
	#data = np.array(data)

	rng = np.random.default_rng(seed)      # random sequence
	# Random starting position Formula : rng.integers(low, high, size)
	offsets  = rng.integers(               # offsets = [12, 13]  =  data[12:16]
	0,
	len(data) - block_size,
	size=batch_size
	)

	x = []
	y = []

	for i in offsets:
		x.append(data[i : i + block_size]) 
		y.append(data[i + 1 : i + 1 + block_size])
		# print("x:",x)
		# print("y:",y)
		# print(f"when input is {x}, the target is {y}")

	return  np.stack([x, y])  # combine the two list into one list 

data = np.arange(20)
out = get_batch(data, block_size=4, batch_size=2, seed=0)
print(out.tolist())

[[[13, 14, 15, 16], [10, 11, 12, 13]], [[14, 15, 16, 17], [11, 12, 13, 14]]]


***Token Embedding Lookup Table***

In [ ]:
import numpy as np

def token_embedding_lookup(vocab_size: int, embed_dim: int, token_ids: list, seed: int = 0) -> list:
	"""
	Build a random embedding table of shape (vocab_size, embed_dim) using
	np.random.default_rng(seed).standard_normal(...), then return the rows
	corresponding to token_ids as a nested list.
	"""
	rng = np.random.default_rng(seed)

	embedding = rng.standard_normal((vocab_size, embed_dim)) 

	print("Shape :",embedding.shape)
	print("Embedding table: \n",embedding), print()

	return embedding[token_ids].tolist()

out = token_embedding_lookup(5, 3, [0, 2, 4], seed=0)    
print(np.round(out, 4).tolist())

Shape : (5, 3)
Embedding table: 
 [[ 0.12573022 -0.13210486  0.64042265]
 [ 0.10490012 -0.53566937  0.36159505]
 [ 1.30400005  0.94708096 -0.70373524]
 [-1.26542147 -0.62327446  0.04132598]
 [-2.32503077 -0.21879166 -1.24591095]]

[[0.1257, -0.1321, 0.6404], [1.304, 0.9471, -0.7037], [-2.325, -0.2188, -1.2459]]


***Numerically Stable Cross-Entropy***

In [ ]:
# this code will give the same results for torch.nn.functional.cross_entropy(logits, targets)

import torch

def cross_entropy(logits, targets):
	# Get the logit corresponding to the correct class
	target_logits = logits.gather(1, targets.unsqueeze(1)).squeeze(1)

	# Compute log(sum(exp(logits))) in a numerically stable way
	log_normalizer = torch.logsumexp(logits, dim=-1)

	# Compute the loss for each example
	loss = -target_logits + log_normalizer

	# Return the mean loss across the batch
	return loss.mean()

logits = torch.tensor([[2.0, 1.0, 0.0]])
targets = torch.tensor([0])
print(round(cross_entropy(logits, targets).item(), 4))


0.4076


In [ ]:

import torch

def cross_entropy(logits, targets):
	
	log_z = torch.logsumexp(logits, dim=1)

	correct_logits = logits[
		torch.arange(logits.shape[0]),   # in this example it is : tensor([0, 1, 2]), so it will be : logits[0,3] --> logits[1,1] --> logits[2,2]
		targets
	]

	loss = -correct_logits + log_z

	return loss.mean()


logits = torch.tensor([
	[2.0, 1.0, 0.0, 3.0],   # example 0
	[1.0, 4.0, 2.0, 0.5],   # example 1
	[0.2, 0.1, 3.0, 1.0]    # example 2
])
targets = torch.tensor([3, 1, 2])
print(round(cross_entropy(logits, targets).item(), 4))


0.2864


In [5]:
torch.arange(logits.shape[0])

tensor([0, 1, 2])

# Numerically Stable Cross-Entropy

Let's give an example so we can understand it more clearly:

Let's assume that:

```text
logits =
[
	[2.0, 1.0, 0.0, 3.0],   ← example 1
	[1.0, 4.0, 2.0, 0.5],   ← example 2
	[0.2, 0.1, 3.0, 1.0]    ← example 3
]

targets = [3, 1, 2]
```

So that means:

```text
example 1 → correct class = 3
example 2 → correct class = 1
example 3 → correct class = 2
```

And for each example we want to find the loss:

```text
loss = -log(probability_of_correct_class)
```

For example:

```text
probabilities = [0.1, 0.2, 0.3, 0.4]
target = 3
```

So the probability of the correct class is `0.4`, therefore:

```text
loss = -log(0.4)
```

But we have **logits** as numbers, not probabilities, like:

```text
[2.0, 1.0, 0.0, 3.0]
```

So first, we need to convert the logits into probabilities using **Softmax**:

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

Where:

```text
zi = the logit of the current class
zj = all logits
```

For example, if:

```text
logits = [2.0, 1.0, 0.0]
```

then Softmax calculates:

```text
e^2
e^1
e^0
```

and divides each one by their total:

```text
	e^2
-----------------
e^2 + e^1 + e^0
```

```text
	e^1
-----------------
e^2 + e^1 + e^0
```

```text
	e^0
-----------------
e^2 + e^1 + e^0
```

This gives us the probabilities.

---


## The Problem With Large Logits

But we need to be careful.

What if we have very large logits like:

```text
logits = [1000, 999, 998]
```

If we directly calculate:

```text
exp(1000)
```

the number is extremely large:

```text
exp(1000) → inf
```

Then our Softmax becomes something like:

```text
inf / (inf + inf + inf)
```

which can result in:

```text
NaN
```

So our implementation would become numerically unstable.

---

## The Max Subtraction Trick

To solve this problem, we can use **Max Subtraction**.

Basically, we first find the largest value in the logits.

For:

```text
logits = [1000, 999, 998]
```

the maximum value is:

```text
m = 1000
```

Then we subtract this value from every logit:

```text
[1000, 999, 998] - 1000
```

which gives:

```text
[0, -1, -2]
```

Now we can safely calculate the exponential:

```text
exp([0, -1, -2])
```

which gives approximately:

```text
[1, 0.3679, 0.1353]
```

Now there is no overflow because all the values are less than or equal to `0`.

---

## Why Can We Subtract the Maximum?

An important question is:

**Does subtracting the maximum change the Softmax probabilities?**

No.

We subtract the same value from every logit, so the effect cancels out.

Mathematically:

$$
\frac{e^{z_i}}{\sum_j e^{z_j}}
=
\frac{e^{z_i-m}}{\sum_j e^{z_j-m}}
$$

So:

```text
[1000, 999, 998]
```

and:

```text
[0, -1, -2]
```

produce exactly the same Softmax probabilities.

The difference is that the second version is much safer to calculate.

---

## Cross-Entropy Formula

Now we know that Cross-Entropy is:

```text
loss = -log(probability_of_correct_class)
```

And Softmax gives us:

$$
p_y = \frac{e^{z_y}}{\sum_j e^{z_j}}
$$

So:

$$
-\log(p_y)
$$

becomes:

$$
-\log\left(\frac{e^{z_y}}{\sum_j e^{z_j}}\right)
$$

Using the logarithm rules:

$$
-\log(e^{z_y}) + \log\left(\sum_j e^{z_j}\right)
$$

which gives:

$$
loss = -z_y + \log\sum_j e^{z_j}
$$

This is very useful because now we don't need to calculate the Softmax probabilities first.

We can calculate the Cross-Entropy directly from the logits.

---

## Using `torch.logsumexp`

The problem is that this part:

```text
log(sum(exp(logits)))
```

can still be dangerous if we calculate it manually.

For example:

```python
torch.exp(logits).sum().log()
```

can overflow for very large logits.

PyTorch provides a function called:

```python
torch.logsumexp()
```

which calculates:

```text
log(sum(exp(x)))
```

in a **numerically stable way**.

So instead of doing:

```python
torch.exp(logits).sum().log()
```

we can simply do:

```python
torch.logsumexp(logits, dim=-1)
```

The `dim=-1` means that we calculate it across the **class dimension**.

For example, if:

```text
logits.shape = (N, C)
```

then:

```python
torch.logsumexp(logits, dim=-1)
```

returns:

```text
shape = (N,)
```

So we get one value for each example.

---

## Getting the Correct Class Logit

We also need:

```text
z_y
```

which means the logit corresponding to the correct target class.

For example:

```text
logits =
[
	[2, 1, 0],
	[1, 4, 2],
	[3, 0, 1]
]

targets = [0, 1, 2]
```

We want:

```text
example 1 → logits[0, 0] → 2
example 2 → logits[1, 1] → 4
example 3 → logits[2, 2] → 1
```

So the result should be:

```text
[2, 4, 1]
```

We can use `gather()` to get these values.

First, our targets have shape:

```text
targets.shape = (N,)
```

For example:

```text
[0, 1, 2]
```

We use:

```python
targets.unsqueeze(1)
```

to change the shape from:

```text
(N,)
```

to:

```text
(N, 1)
```

So:

```text
[0, 1, 2]
```

becomes:

```text
[
	[0],
	[1],
	[2]
]
```

Then we can use:

```python
logits.gather(1, targets.unsqueeze(1))
```

This gives:

```text
[
	[2],
	[4],
	[1]
]
```

We use:

```python
.squeeze(1)
```

to remove the extra dimension:

```text
[2, 4, 1]
```

So finally:

```python
target_logits = logits.gather(1, targets.unsqueeze(1)).squeeze(1)
```

---

## Putting Everything Together

From the Cross-Entropy formula:

$$
loss = -z_y + \log\sum_j e^{z_j}
$$

we already have the target logits:

```python
target_logits = logits.gather(1, targets.unsqueeze(1)).squeeze(1)
```

And we can calculate the stable log-normalizer using:

```python
log_normalizer = torch.logsumexp(logits, dim=-1)
```

So the loss for each example is:

```python
loss = -target_logits + log_normalizer
```

This gives us one loss value for every example.

Finally, we need the **mean loss across the batch**:

```python
loss.mean()
```

---


## Example

Let's test it with the example from the exercise:

```python
logits = torch.tensor([[2.0, 1.0, 0.0]])
targets = torch.tensor([0])

loss = cross_entropy(logits, targets)

print(loss)
```

The result should be approximately:

```text
tensor(0.4076)
```

This matches the expected Cross-Entropy.

---

## Testing With Very Large Logits

We can also test our implementation with very large logits:

```python
logits = torch.tensor([[1000.0, 999.0, 998.0]])
targets = torch.tensor([0])

loss = cross_entropy(logits, targets)

print(loss)
```

Even though the logits are very large, our implementation should still return a normal finite value instead of:

```text
inf
```

or:

```text
nan
```

The reason is that `torch.logsumexp()` uses a numerically stable calculation internally.

---

## Final Idea

The main idea of **Numerically Stable Cross-Entropy** is:

```text
Large logits
	 ↓
Avoid directly calculating exp(large numbers)
	 ↓
Use torch.logsumexp()
	 ↓
Get the logit of the correct class
	 ↓
loss = -target_logit + logsumexp
	 ↓
Take the mean
```

So the most important formula to remember is:

$$
\boxed{
CrossEntropy = -target\_logit + logsumexp(logits)
}
$$

This allows us to calculate Cross-Entropy directly from the logits in a numerically stable way, without explicitly calculating the Softmax probabilities first.


***Greedy Autoregressive Text Generation***

In [6]:
import numpy as np

def generate_greedy(model, idx: list, max_new_tokens: int, context_size: int) -> list:

    idx = np.array([idx])

    for _ in range(max_new_tokens):

        # Crop to the last context_size tokens
        idx_cond = idx[:, -context_size:]

        # Get logits from the model
        logits = model(idx_cond)

        # Take logits from the last time step
        logits = logits[:, -1, :]

        # Pick the token with the highest logit
        next_token = np.argmax(logits, axis=-1)

        # Append the new token
        idx = np.concatenate([idx, next_token[:, None]], axis=1)

    return idx[0].tolist()


def model(x):
	V = 5
	T = x.shape[1]
	logits = np.zeros((1, T, V))
	for t in range(T):
		nxt = (int(x[0, t]) + 1) % V
		logits[0, t, nxt] = 1.0
	return logits
print(generate_greedy(model, [0], 4, 8))

[0, 1, 2, 3, 4]


***Implement AdamW Optimizer Step***

In [ ]:
import numpy as np

def adamw_update(w, g, m, v, t, lr, beta1, beta2, epsilon, weight_decay):
		"""
		Perform one AdamW optimizer step.
		Args:
			w: parameter vector (np.ndarray)
			g: gradient vector (np.ndarray)
			m: first moment vector (np.ndarray)
			v: second moment vector (np.ndarray)
			t: integer, current time step
			lr: float, learning rate
			beta1: float, beta1 parameter
			beta2: float, beta2 parameter
			epsilon: float, small constant
			weight_decay: float, weight decay coefficient
		Returns:
			w_new, m_new, v_new
		"""
		# Your code here
		pass

***Causal Cumulative Mean via Triangular Matrix Multiply***

In [ ]:
import numpy as np

def causal_cumulative_mean(X: np.ndarray) -> np.ndarray:
	"""
	Args:
		X: numpy array of shape (B, T, C)
	Returns:
		numpy array of shape (B, T, C) where output[b, t] is the mean of X[b, :t+1]
		along the time dimension, computed via a lower-triangular weight matrix and
		a batched matrix multiplication.
	"""
	pass


***Softmax Activation Function Implementation***

In [ ]:
import math
import numpy as np 
def softmax(scores):
	pass

***Learned Positional Embeddings***

In [ ]:
import numpy as np

def learned_positional_encoding(token_embeddings: np.ndarray, position_embedding_table: np.ndarray, start_pos: int = 0) -> np.ndarray:
	"""
	Apply learned positional embeddings to token embeddings.
	
	Args:
		token_embeddings: (batch_size, seq_len, d_model) array of token embeddings
		position_embedding_table: (max_seq_len, d_model) learned positional embedding lookup table
		start_pos: Starting position index (default 0)
	
	Returns:
		Array of shape (batch_size, seq_len, d_model) with positional information applied
	"""
	pass

***Implement Self-Attention Mechanism***

In [ ]:
import numpy as np

def compute_qkv(X, W_q, W_k, W_v):
	"""Compute Query, Key, Value matrices from input X and weight matrices."""
	Q = np.dot(X, W_q)
	K = np.dot(X, W_k)
	V = np.dot(X, W_v)
	return Q, K, V

def self_attention(Q, K, V):
	"""
	Compute scaled dot-product self-attention.
	
	Args:
		Q: Query matrix of shape (seq_len, d_k)
		K: Key matrix of shape (seq_len, d_k)
		V: Value matrix of shape (seq_len, d_v)
	
	Returns:
		Attention output of shape (seq_len, d_v)
	"""
	# Your code here
	pass


***Build Scaled Dot-Product Attention***

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask: np.ndarray = None) -> tuple:
	"""
	Compute Scaled Dot-Product Attention.
	
	Args:
		Q: Query matrix of shape (seq_len_q, d_k)
		K: Key matrix of shape (seq_len_k, d_k)
		V: Value matrix of shape (seq_len_k, d_v)
		mask: Optional binary mask of shape (seq_len_q, seq_len_k)
	
	Returns:
		Tuple of (output, attention_weights)
	"""
	# Your code here
	pass

***Implement Masked Self-Attention***

In [ ]:
import numpy as np

def compute_qkv(X: np.ndarray, W_q: np.ndarray, W_k: np.ndarray, W_v: np.ndarray):
	"""
	Compute Query (Q), Key (K), and Value (V) matrices.
	"""
	return np.dot(X, W_q), np.dot(X, W_k), np.dot(X, W_v)

def masked_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask: np.ndarray) -> np.ndarray:
	"""
	Compute masked self-attention.
	"""
	# Your code here
	pass

***Implement Multi-Head Attention***

In [ ]:
import numpy as np
from typing import Tuple

def compute_qkv(X: np.ndarray, W_q: np.ndarray, W_k: np.ndarray, W_v: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
	"""
	Compute Query, Key, and Value matrices.
	
	Args:
		X: Input matrix of shape (seq_len, d_model)
		W_q, W_k, W_v: Weight matrices of shape (d_model, d_model)
	
	Returns:
		Q, K, V matrices each of shape (seq_len, d_model)
	"""
	# Your code here
	pass

def self_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray) -> np.ndarray:
	"""
	Compute scaled dot-product self-attention.
	
	Args:
		Q: Query matrix of shape (seq_len, d_k)
		K: Key matrix of shape (seq_len, d_k)
		V: Value matrix of shape (seq_len, d_k)
	
	Returns:
		Attention output of shape (seq_len, d_k)
	"""
	# Your code here
	pass

def multi_head_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray, n_heads: int) -> np.ndarray:
	"""
	Compute multi-head attention.
	
	Args:
		Q, K, V: Matrices of shape (seq_len, d_model)
		n_heads: Number of attention heads
	
	Returns:
		Attention output of shape (seq_len, d_model)
	"""
	# Your code here
	pass

***Implement Position-wise Feed-Forward Block with Residual and Dropout***

In [ ]:
import numpy as np

def ffn(x: list[float], W1: list[list[float]], b1: list[float], W2: list[list[float]], b2: list[float], dropout_p: float=0.1, seed: int=42) -> list[float]:
	"""
	Implement a position-wise feed-forward block with residual and dropout.

	Args:
		x: input vector
		W1, b1: first linear layer parameters
		W2, b2: second linear layer parameters
		dropout_p: dropout probability
		seed: random seed for reproducibility

	Returns:
		Output vector after FFN block (rounded to 4 decimals)
	"""
	# Your code here
	pass

***Implement a Simple Residual Block with Shortcut Connection***

In [ ]:
import numpy as np

def residual_block(x: np.ndarray, w1: np.ndarray, w2: np.ndarray) -> np.ndarray:
	# Your code here
	pass

***Implement LayerNorm from Scratch***

In [ ]:
import torch

def layer_norm(x, gamma, beta, eps=1e-5):
	# TODO: normalize over the last dim, then affine-transform with gamma and beta
	pass

***Implement Dropout from Scratch***

In [ ]:
import torch

def dropout(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
	# TODO: implement inverted dropout
	pass


***Build a Transformer Encoder Layer***

In [ ]:
import numpy as np

def transformer_encoder_layer(X: np.ndarray, weights: dict, num_heads: int, eps: float = 1e-5) -> np.ndarray:
	"""
	Forward pass of a single Transformer Encoder Layer.

	Args:
		X: Input tensor of shape (batch_size, seq_len, d_model)
		weights: Dictionary containing all weight matrices and normalization parameters
		num_heads: Number of attention heads
		eps: Epsilon for layer normalization

	Returns:
		Output tensor of shape (batch_size, seq_len, d_model)
	"""
	# Your code here
	pass